In [1]:
import os.path as op

import numpy as np

import yaml
from FAM.processing import load_exp_settings, preproc_mridata, preproc_behdata
from FAM.visualize.beh_viewer import BehViewer

# load settings from yaml
with open('exp_params.yml', 'r') as f_in:
    params = yaml.safe_load(f_in)

In [2]:
# access parser options
sj = 'all'
system_dir = 'local'
data_type = 'beh'
exclude_sj = ['002'] # list of excluded subjects
task = 'FA'
use_atlas = None
T2_file = False

In [ ]:
## Load data object --> as relevant paths, variables and utility functions
print("Loading {data} data for subject {sj}!".format(data=data_type, sj=sj))

FAM_data = load_exp_settings.MRIData(params, sj, 
                                    repo_pth = op.split(load_exp_settings.__file__)[0], 
                                    base_dir = system_dir, exclude_sj = exclude_sj)

print('Subject list to vizualize is {l}'.format(l=str(FAM_data.sj_num)))


In [4]:
## Load preprocessing class for each data type ###

# get behavioral info 
FAM_beh = preproc_behdata.PreprocBeh(FAM_data)
# and mri info
FAM_mri = preproc_mridata.PreprocMRI(FAM_data)

In [5]:
## initialize plotter object
plotter = BehViewer(FAM_data)

In [ ]:
# make dataframe with behavioral results
att_RT_df = FAM_beh.get_FA_behavioral_results(participant_list = FAM_data.sj_num,
                                            ses_type = 'func')

# get accuracy per ecc
acc_df = FAM_beh.get_FA_accuracy(att_RT_df = att_RT_df)

In [10]:
import seaborn as sns
import matplotlib.pyplot as plt

# set font type for plots globally
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Helvetica'

In [ ]:
_ = plotter.plot_FA_RTecc(att_RT_df = att_RT_df, filename = None, figsize = (8,5), ecc_colors=['#006e7f', '#f8cb2e', '#ee5007'])

In [ ]:
## test in repeated measures anova

from statsmodels.stats.anova import AnovaRM

RT_anova = AnovaRM(att_RT_df[att_RT_df['correct'] == 1], 
                depvar = 'RT',
                subject = 'sj',
                within = ['bar_ecc_deg', 'unatt_bar_ecc_deg'],
                aggregate_func='mean'
                ).fit()
print(RT_anova)

In [ ]:
_ = plotter.plot_FA_ACCecc(acc_df = acc_df, per_pp = False)

In [ ]:
## test in repeated measures anova

acc_anova = AnovaRM(acc_df[acc_df['correct'] == 1], 
                depvar='accuracy',
                subject='sj',
                within=['bar_ecc_deg', 'unatt_bar_ecc_deg'],
                aggregate_func='mean'
                ).fit()
print(acc_anova)

In [ ]:
_ = plotter.plot_FA_RTdist(att_RT_df = att_RT_df, ecc_colors=['#006e7f', '#f8cb2e', '#ee5007'])

In [ ]:
## calculate ANOVA RT vs distance PER ECC
for ecc_val in np.sort(att_RT_df.bar_ecc_deg.unique()):

    print('Target bar ecc %.2f'%ecc_val)
    
    new_RT_anova = AnovaRM(att_RT_df[(att_RT_df['correct'] == 1) &\
                                   (att_RT_df['bar_ecc_deg'] == ecc_val) &\
                                   (att_RT_df['bars_pos'] == 'parallel')], 
                    depvar = 'RT',
                    subject = 'sj',
                    within = ['interbar_dist_deg'],
                    aggregate_func='mean'
                    ).fit()
    print(new_RT_anova)

In [ ]:
## LMM to check effect of bar ecc and distance

from statsmodels.formula.api import mixedlm

model = mixedlm(formula = 'np.log(RT) ~ bar_ecc_deg * interbar_dist_deg',
                data = att_RT_df[(att_RT_df['correct'] == 1) &\
                                (att_RT_df['bars_pos'] == 'parallel')], 
                groups = 'sj').fit()
print(model.summary())

In [ ]:
# also check ecc

model = mixedlm(formula = 'np.log(RT) ~ bar_ecc_deg * unatt_bar_ecc_deg',
                #re_formula = '~ unatt_bar_ecc_deg',
                data = att_RT_df[(att_RT_df['correct'] == 1)], 
                groups = 'sj').fit()
print(model.summary())

In [ ]:
### check distance effect for accuracy

# get accuracy per distance
acc_dist_df = FAM_beh.get_FA_accuracy(att_RT_df = att_RT_df[att_RT_df['bars_pos'] == 'parallel'], 
                                      condition = 'interbar_dist_deg')

## calculate ANOVA acc vs distance PER ECC
for ecc_val in acc_dist_df.bar_ecc_deg.unique():

    print('Target bar ecc %.2f'%ecc_val)
    
    new_acc_anova = AnovaRM(acc_dist_df[(acc_dist_df['correct'] == 1) &\
                                        (acc_dist_df['bar_ecc_deg'] == ecc_val)], 
                    depvar='accuracy',
                    subject='sj',
                    within=['interbar_dist_deg'],
                    aggregate_func='mean'
                    ).fit()
    print(new_acc_anova)

In [ ]:
_ = plotter.plot_FA_ACCdist(acc_df = acc_dist_df, ecc_colors=['#006e7f', '#f8cb2e', '#ee5007'])

In [27]:
# ## save RT and accuracy values to test in R

# # set path to save file
# beh_dir = op.join(FAM_beh.MRIObj.derivatives_pth, 'behavioral')
# os.makedirs(beh_dir, exist_ok=True)

# ## save accuracy for parallel trials
# att_RT_df[att_RT_df['bars_pos'] == 'parallel'].to_csv(op.join(beh_dir, 'df_accuracy_parallel_group.csv'))

# ## save RT for parallel trials
# att_RT_df[(att_RT_df['bars_pos'] == 'parallel') &\
#         (att_RT_df['correct'] == 1)].to_csv(op.join(beh_dir, 'df_RT_parallel_group.csv'))

